# PDF Question Answering with Astra DB and Modern LangChain

This notebook updates the older Cassandra/CassIO tutorial to the current `langchain-astradb` integration.

**Flow:** PDF → text extraction → chunking → OpenAI embeddings → Astra DB vector store → similarity retrieval → ChatOpenAI answer.

> Before running: create an Astra DB Serverless database with Vector Search enabled. From the Astra DB console, copy the **API Endpoint** and create an application token. You need the API endpoint (not only the database ID).

## 1. Install dependencies

In [5]:
%pip install -U langchain-core langchain-openai langchain-astradb langchain-text-splitters openai pypdf datasets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip install dotenv


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Import packages

In [5]:
import os
from getpass import getpass

from pypdf import PdfReader
from langchain_astradb import AstraDBVectorStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os

## 3. Configure credentials

Enter credentials when prompted. Do not store API tokens or keys directly in notebook cells.

**Security:** The original notebook contained hard-coded Astra DB and OpenAI credentials. Revoke/rotate those credentials in their respective consoles before continuing, then enter the newly generated values below.

In [9]:
from dotenv import load_dotenv
load_dotenv()
ASTRA_DB_API_ENDPOINT = os.getenv("ASTRA_DB_API_ENDPOINT")
ASTRA_DB_APPLICATION_TOKEN = os.getenv("ASTRA_DB_APPLICATION_TOKEN")
ASTRA_DB_ID =os.getenv("ASTRA_DB_ID")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Optional: set a namespace if your Astra DB database uses one.
ASTRA_DB_NAMESPACE = None  # Example: "default_keyspace" 

## 4. Read the PDF and extract text

In [10]:
PDF_PATH = "budget_speech.pdf"  # Keep the PDF in the notebook's working directory.

pdf_reader = PdfReader(PDF_PATH)

pages_text = []
for page_number, page in enumerate(pdf_reader.pages, start=1):
    page_text = page.extract_text() or ""
    if page_text.strip():
        pages_text.append(f"Page {page_number}\n{page_text}")

raw_text = "\n\n".join(pages_text)

print(f"Extracted {len(raw_text):,} characters from {len(pdf_reader.pages)} pages.")
print(raw_text[:1000])

Extracted 102,762 characters from 65 pages.
Page 1
GOVERNMENT OF INDIA
BUDGET 2026-2027
SPEECH
OF
NIRMALA SITHARAMAN
MINISTER OF FINANCE
February 1,  2026

Page 3
 
CONTENTS 
 
Page No. 
Introduction 1 
                                                          PART - A                                          
Yuva Shakti and 3 kartavya 2 
Reform Express 3 
First kartavya: to accelerate and sustain economic growth  3 
Second kartavya: fulfil aspirations and build capacity 10 
Third kartavya: Sabka Sath, Sabka Vikas 14  
16th Finance Commission 18 
Fiscal Consolidation 18 
 
PART – B 
Direct taxes 20 
Indirect Taxes  26 
 
Annexure to Part-A 32 
Annexure to Part-B 
Amendments relating to Direct Taxes 33 
Amendments relating to Indirect Taxes 50 
 
 
  

Page 5
 
 
Budget 2026-2027 
 
Speech of 
Nirmala Sitharaman 
Minister of Finance 
February 1, 2026 
 
Hon’ble Speaker, 
On the sacred occasion of Magha Purnima and the birth 
anniversary of Guru Ravidas, I present the Budget for the yea

## 5. Split the PDF text into chunks

In [11]:
text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=800,
    chunk_overlap=200,
    length_function=len,
)

documents = text_splitter.create_documents(
    [raw_text],
    metadatas=[{"source": PDF_PATH}]
)

print(f"Created {len(documents)} chunks.")
print(documents[0].page_content[:500])

Created 171 chunks.
Page 1
GOVERNMENT OF INDIA
BUDGET 2026-2027
SPEECH
OF
NIRMALA SITHARAMAN
MINISTER OF FINANCE
February 1,  2026
Page 3
 
CONTENTS 
 
Page No. 
Introduction 1 
                                                          PART - A                                          
Yuva Shakti and 3 kartavya 2 
Reform Express 3 
First kartavya: to accelerate and sustain economic growth  3 
Second kartavya: fulfil aspirations and build capacity 10 
Third kartavya: Sabka Sath, Sabka Vikas 14  
16th Finance Commis


## 6. Create the Astra DB vector store

`AstraDBVectorStore` replaces the old `Cassandra` LangChain class. This modern integration connects using the Astra DB API endpoint and application token; you do not need `cassio.init()` for this approach.

A new collection name is used to avoid mixing vectors created by a previous embedding configuration.

In [12]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

astra_vector_store = AstraDBVectorStore(
    collection_name="budget_speech_rag_v2",
    embedding=embeddings,
    api_endpoint=ASTRA_DB_API_ENDPOINT,
    token=ASTRA_DB_APPLICATION_TOKEN,
    namespace=ASTRA_DB_NAMESPACE,
)

print("Astra DB vector store is ready.")

Astra DB vector store is ready.


## 7. Insert document chunks into Astra DB

In [13]:
# Add chunks in batches. Re-running this cell may insert duplicate chunks.
BATCH_SIZE = 50

for start in range(0, len(documents), BATCH_SIZE):
    batch = documents[start:start + BATCH_SIZE]
    astra_vector_store.add_documents(batch)
    print(f"Inserted chunks {start + 1} to {start + len(batch)}")

print(f"Finished inserting {len(documents)} chunks.")

Inserted chunks 1 to 50
Inserted chunks 51 to 100
Inserted chunks 101 to 150
Inserted chunks 151 to 171
Finished inserting 171 chunks.


## 8. Configure the chat model and RAG prompt

The old `VectorStoreIndexWrapper.query()` call is replaced with explicit retrieval and generation. This makes the RAG steps visible and easier to customize.

In [14]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

retriever = astra_vector_store.as_retriever(
    search_kwargs={"k": 4}
)

rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Answer the question using only the supplied context. "
        "If the context does not contain the answer, say you do not know. "
        "Keep the answer concise."
    ),
    (
        "human",
        "Context:\n{context}\n\nQuestion: {question}"
    ),
])

## 9. Ask questions

In [15]:
import time

def answer_question(question: str) -> str:

    # Step 1: Retrieve documents
    print("\n1. Searching Astra DB...", flush=True)
    start = time.perf_counter()

    retrieved_docs = retriever.invoke(question)

    print(f"Retrieval completed in {time.perf_counter() - start:.2f} seconds", flush=True)

    if not retrieved_docs:
        return "I could not find relevant information in the PDF."

    # Step 2: Display retrieved documents immediately
    print("\n2. Retrieved Documents:", flush=True)

    for index, doc in enumerate(retrieved_docs, start=1):
        print(f"\n{index}. Source: {doc.metadata.get('source', 'Unknown')}")
        print(doc.page_content[:300].replace("\n", " "))

    # Step 3: Prepare context
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)

    messages = rag_prompt.format_messages(
        context=context,
        question=question,
    )

    # Step 4: Generate answer
    print("\n3. Generating answer using LLM...", flush=True)
    start = time.perf_counter()

    response = llm.invoke(messages)

    print(f"LLM completed in {time.perf_counter() - start:.2f} seconds", flush=True)

    return response.content


while True:
    query_text = input("\nEnter your question (or type 'quit' to exit): ").strip()

    if query_text.lower() == "quit":
        break

    if not query_text:
        continue

    print("\nANSWER:")
    print(answer_question(query_text))

## 10. Inspect similarity scores (optional)

In [16]:
query_text = "How much will the agriculture credit target increase?"

matches = astra_vector_store.similarity_search_with_score(query_text, k=4)

for doc, score in matches:
    print(f"Score: {score:.4f}")
    print(doc.page_content[:500])
    print("-" * 80)

Score: 0.7022
78. High Value Agriculture : To diversify farm outputs, increase 
productivity, enhance farmers’ incomes, and create new employment 
opportunities, we will support high value crops such as coconut, 
sandalwood, cocoa and cashew in our coastal areas. Agar trees in North 
East and nuts such as, almonds, walnuts and pine nuts in our hilly regions 
will also be supported. 
79. India is the world’s largest producer of coconuts.  
About 30 million people, including nearly 10 million farmers, depend on
--------------------------------------------------------------------------------
Score: 0.7022
78. High Value Agriculture : To diversify farm outputs, increase 
productivity, enhance farmers’ incomes, and create new employment 
opportunities, we will support high value crops such as coconut, 
sandalwood, cocoa and cashew in our coastal areas. Agar trees in North 
East and nuts such as, almonds, walnuts and pine nuts in our hilly regions 
will also be supported. 
79. India is the w

In [17]:
print(matches)

[(Document(id='3f7ca578380845b4a83c01f049bc0e39', metadata={'source': 'budget_speech.pdf'}, page_content='78. High Value Agriculture : To diversify farm outputs, increase \nproductivity, enhance farmers’ incomes, and create new employment \nopportunities, we will support high value crops such as coconut, \nsandalwood, cocoa and cashew in our coastal areas. Agar trees in North \nEast and nuts such as, almonds, walnuts and pine nuts in our hilly regions \nwill also be supported. \n79. India is the world’s largest producer of coconuts.  \nAbout 30 million people, including nearly 10 million farmers, depend on \ncoconuts for their livelihood. To further enhance competitiveness in \ncoconut production, I propose a Coconut Promotion Scheme to increase \nproduction and enhance productivity through various interventions \nPage 20\n 16  \n \nincluding replacing old and non -productive trees with new'), 0.7021554), (Document(id='f740f76b965441e08a5d9269be3476ac', metadata={'source': 'budget_spee

## Old API → modern API mapping

| Old tutorial | Updated notebook |
|---|---|
| `langchain.vectorstores.cassandra.Cassandra` | `langchain_astradb.AstraDBVectorStore` |
| `cassio.init(token=..., database_id=...)` | Pass `api_endpoint` and `token` to `AstraDBVectorStore` |
| `langchain.indexes.vectorstore.VectorStoreIndexWrapper` | `vector_store.as_retriever()` plus explicit LLM call |
| `langchain.llms.OpenAI` | `langchain_openai.ChatOpenAI` |
| `langchain.embeddings.OpenAIEmbeddings` | `langchain_openai.OpenAIEmbeddings` |
| `langchain.text_splitter.CharacterTextSplitter` | `langchain_text_splitters.CharacterTextSplitter` |

### Notes
- `ASTRA_DB_API_ENDPOINT` is the HTTPS API endpoint shown in the Astra DB database details. It is not the database UUID.
- `ASTRA_DB_NAMESPACE` can be left as `None` if you want the database default namespace.
- The first insertion creates/uses the named collection. Re-running insertion can create duplicate documents.
- If you change the embedding model, use a fresh collection or re-embed all stored documents consistently.
- The notebook uses `budget_speech.pdf` as the input file; place it beside this notebook or update `PDF_PATH`.